# Stage 5: Feature Engineering & Selection

Extracting features from pretrained ResNet-50, applying PCA, and analyzing feature importance.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models, transforms
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ============================================================
# 5.1 Feature Extractor from Pretrained ResNet-50
# ============================================================
class FeatureExtractor(nn.Module):
    """
    Extracts features from the penultimate layer of ResNet-50.
    Output: 2048-dimensional feature vector per image.
    """
    def __init__(self):
        super().__init__()
        backbone = models.resnet50(weights='IMAGENET1K_V2')
        # Remove the final classification layer
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.features.eval()
    
    def forward(self, x):
        with torch.no_grad():
            feat = self.features(x)  # (B, 2048, 1, 1)
            feat = feat.squeeze(-1).squeeze(-1)  # (B, 2048)
        return feat


extractor = FeatureExtractor().to(DEVICE)
print('FeatureExtractor loaded (ResNet-50 backbone, 2048-dim output)')

In [ ]:
# ============================================================
# 5.2 Extract Features from Dataset
# ============================================================
def extract_features(loader, extractor, device):
    all_features, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        feats = extractor(imgs)
        all_features.append(feats.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.vstack(all_features), np.array(all_labels)

print('Function defined. Run with dataloaders from notebook 03.')

# Example usage:
# train_features, train_labels = extract_features(train_loader, extractor, DEVICE)
# print(f'Feature matrix shape: {train_features.shape}')  # (N, 2048)

In [ ]:
# ============================================================
# 5.3 PCA — Dimensionality Reduction
# ============================================================
def apply_pca_and_visualize(features, labels, n_components=50):
    print(f'Applying PCA: {features.shape[1]} → {n_components} dimensions')
    
    pca = PCA(n_components=n_components, random_state=42)
    features_pca = pca.fit_transform(features)
    
    # Explained variance
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scree plot
    axes[0].plot(range(1, n_components+1), pca.explained_variance_ratio_, 'bo-', markersize=4)
    axes[0].plot(range(1, n_components+1), cumvar, 'r-', label='Cumulative')
    axes[0].axhline(0.95, color='gray', linestyle='--', label='95% threshold')
    axes[0].set_title('PCA Explained Variance')
    axes[0].set_xlabel('Principal Component')
    axes[0].set_ylabel('Explained Variance Ratio')
    axes[0].legend()
    
    # t-SNE on PCA features
    print('Running t-SNE (this may take ~2min)...')
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    features_2d = tsne.fit_transform(features_pca)
    
    colors = ['#2ecc71' if l == 0 else '#e74c3c' for l in labels]
    scatter = axes[1].scatter(features_2d[:, 0], features_2d[:, 1],
                              c=colors, alpha=0.7, s=20, edgecolors='none')
    axes[1].set_title('t-SNE of ResNet-50 Features')
    axes[1].set_xlabel('t-SNE 1')
    axes[1].set_ylabel('t-SNE 2')
    
    import matplotlib.patches as mpatches
    axes[1].legend(handles=[
        mpatches.Patch(color='#2ecc71', label='Normal'),
        mpatches.Patch(color='#e74c3c', label='Defective')
    ])
    
    plt.tight_layout()
    plt.savefig('../assets/feature_tsne.png', dpi=150)
    plt.show()
    
    n95 = np.searchsorted(cumvar, 0.95) + 1
    print(f'Components needed for 95% variance: {n95}')
    return features_pca, pca

print('PCA + t-SNE functions defined.')

In [ ]:
# ============================================================
# 5.4 SSIM-based Structural Features (for Autoencoder)
# ============================================================
from scipy.signal import correlate2d

def compute_ssim_features(img1_array, img2_array):
    """
    Simplified SSIM between original and reconstructed image.
    Used as anomaly score in autoencoder branch.
    """
    mu1, mu2 = img1_array.mean(), img2_array.mean()
    sigma1, sigma2 = img1_array.std(), img2_array.std()
    sigma12 = np.mean((img1_array - mu1) * (img2_array - mu2))
    
    C1, C2 = 0.01**2, 0.03**2
    ssim = ((2*mu1*mu2 + C1) * (2*sigma12 + C2)) / \
           ((mu1**2 + mu2**2 + C1) * (sigma1**2 + sigma2**2 + C2))
    return ssim

print('SSIM feature function defined.')
print('This is used to compare original vs reconstructed images from the autoencoder.')
print('Lower SSIM = higher anomaly score.')